# Transfer Learning and Fine-Tuning

## Overview

Transfer learning reuses representations learned from large datasets (ImageNet, large text corpora) for new tasks with limited data. It is the most practical approach to deep learning when training data is scarce.

**Two strategies:**

| Strategy | When | How |
|---|---|---|
| Feature extraction | Very small dataset | Freeze pretrained backbone; train only new head |
| Fine-tuning | Moderate dataset | Unfreeze some/all layers; train with low LR |

**Typical transfer learning workflow:**
```
1. Load pretrained model
2. Replace final classification head
3. Freeze backbone layers
4. Train head for N epochs (high LR)
5. Unfreeze backbone (or top layers)
6. Train entire network at low LR
```

**Why it works:** early layers learn universal features (edges, textures, simple patterns) that transfer across domains. Later layers learn task-specific features that benefit from fine-tuning.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from copy import deepcopy

torch.manual_seed(42)
rng  = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Simulate: small dataset of 3-channel 32x32 spatial sensor maps
# "Source task": classify 4 water quality regimes (pretrained on large dataset)
# "Target task": binary restoration success (small dataset)
def make_img_dataset(n, n_classes, img_size=32, n_channels=3, seed=0):
    rng_l = np.random.default_rng(seed)
    imgs, labels = [], []
    for i in range(n):
        lbl = rng_l.integers(n_classes)
        base = lbl / (n_classes - 1)
        img  = rng_l.normal(base, 0.2, (n_channels, img_size, img_size)).astype(np.float32)
        img  = img.clip(0, 1)
        imgs.append(img); labels.append(lbl)
    return (torch.from_numpy(np.array(imgs)),
            torch.tensor(labels, dtype=torch.long))

# Large source dataset (pretrain)
X_src, y_src = make_img_dataset(n=1200, n_classes=4, seed=1)
# Small target dataset (fine-tune) -- only 200 samples
X_tgt, y_tgt = make_img_dataset(n=200,  n_classes=2, seed=2)
print(f"Source: {X_src.shape}, classes={y_src.unique().tolist()}")
print(f"Target: {X_tgt.shape}, classes={y_tgt.unique().tolist()}")

---
## Pretraining on Source Task

In [ ]:
class ConvBackbone(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),          nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),          nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.feature_dim = 64 * 4 * 4

class FullModel(nn.Module):
    def __init__(self, backbone, n_classes):
        super().__init__()
        self.backbone   = backbone
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4),
            nn.Linear(backbone.feature_dim, 128), nn.ReLU(),
            nn.Linear(128, n_classes))
    def forward(self, x):
        return self.classifier(self.backbone.features(x))

def quick_train(model, X, y, epochs=30, lr=1e-3, batch=32):
    split = int(0.8 * len(X))
    dl = DataLoader(TensorDataset(X[:split], y[:split]), batch_size=batch, shuffle=True)
    Xv, yv = X[split:].to(device), y[split:].to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    best_acc, best_state = 0, None
    for ep in range(epochs):
        model.train()
        for Xb, yb in dl:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            acc = (model(Xv).argmax(1) == yv).float().mean().item()
        if acc > best_acc:
            best_acc = acc; best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return best_acc

backbone = ConvBackbone().to(device)
src_model = FullModel(backbone, n_classes=4).to(device)
acc_src = quick_train(src_model, X_src, y_src, epochs=30)
print(f"Source task accuracy: {acc_src:.3f}")
print("Backbone now contains learned feature representations")

---
## Feature Extraction: Frozen Backbone

In [ ]:
# Strategy 1: freeze backbone, train only new head
pretrained_backbone = deepcopy(src_model.backbone)
# Freeze all backbone parameters
for param in pretrained_backbone.parameters():
    param.requires_grad = False
frozen_model = FullModel(pretrained_backbone, n_classes=2).to(device)
# Only head parameters are trainable
trainable = sum(p.numel() for p in frozen_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in frozen_model.parameters())
print(f"Trainable params (frozen backbone): {trainable:,} / {total:,}")
acc_frozen = quick_train(frozen_model, X_tgt, y_tgt, epochs=40, lr=1e-3)

# Baseline: train from scratch on small target dataset
scratch_backbone = ConvBackbone().to(device)
scratch_model    = FullModel(scratch_backbone, n_classes=2).to(device)
acc_scratch = quick_train(scratch_model, X_tgt, y_tgt, epochs=40, lr=1e-3)

print(f"\nTarget task accuracy comparison:")
print(f"  From scratch:          {acc_scratch:.3f}")
print(f"  Feature extraction:    {acc_frozen:.3f}")

---
## Fine-Tuning: Unfreezing Backbone

In [ ]:
# Strategy 2: fine-tune entire network at low LR
finetune_model = deepcopy(frozen_model)
# Unfreeze all layers
for param in finetune_model.parameters():
    param.requires_grad = True

# Two-phase training: head only, then full fine-tune
# Phase 1: head-only (already done in frozen_model -- reuse)
# Phase 2: unfreeze and train at low LR
split = int(0.8 * len(X_tgt))
dl_ft = DataLoader(TensorDataset(X_tgt[:split], y_tgt[:split]),
                   batch_size=16, shuffle=True)
Xv_t, yv_t = X_tgt[split:].to(device), y_tgt[split:].to(device)
# Use different LR for backbone vs head (discriminative fine-tuning)
opt_ft = optim.Adam([
    {'params': finetune_model.backbone.parameters(), 'lr': 1e-4},  # low LR for backbone
    {'params': finetune_model.classifier.parameters(), 'lr': 5e-4}, # higher LR for head
], weight_decay=1e-4)
crit = nn.CrossEntropyLoss()
best_acc_ft, best_state_ft = 0, None
accs_ft = []
for ep in range(40):
    finetune_model.train()
    for Xb, yb in dl_ft:
        Xb, yb = Xb.to(device), yb.to(device)
        opt_ft.zero_grad(); crit(finetune_model(Xb), yb).backward(); opt_ft.step()
    finetune_model.eval()
    with torch.no_grad():
        acc = (finetune_model(Xv_t).argmax(1) == yv_t).float().mean().item()
    accs_ft.append(acc)
    if acc > best_acc_ft:
        best_acc_ft = acc; best_state_ft = deepcopy(finetune_model.state_dict())
print(f"  Full fine-tuning:      {best_acc_ft:.3f}")
print(f"\nKey: discriminative LRs -- backbone LR < head LR during fine-tuning")

In [ ]:
# Using torchvision pretrained models (production approach)
try:
    from torchvision import models
    # Load ResNet18 pretrained on ImageNet
    resnet = models.resnet18(weights='DEFAULT')
    # Inspect architecture
    print("ResNet18 final layer:", resnet.fc)
    # Replace head for binary classification
    in_features = resnet.fc.in_features
    resnet.fc = nn.Linear(in_features, 2)
    print(f"New head: {resnet.fc}")
    total     = sum(p.numel() for p in resnet.parameters())
    print(f"Total parameters: {total:,}")
    # Freeze backbone, train only new head
    for name, param in resnet.named_parameters():
        if 'fc' not in name:
            param.requires_grad = False
    trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
    print(f"Trainable (head only): {trainable:,}")
    print("\nFor real image data, always use torchvision pretrained models")
    print("Input: 3-channel images normalised with ImageNet mean/std")
    print("  mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")
except ImportError:
    print("pip install torchvision")
    print("torchvision.models provides: ResNet, EfficientNet, ViT, and more")
    print("All support weights='DEFAULT' for pretrained ImageNet weights")

---

## Common Pitfalls

**1. Fine-tuning with the same high learning rate used for the head**  
The pretrained backbone contains carefully learned features. A high learning rate catastrophically overwrites them in the first few updates — a phenomenon called catastrophic forgetting. Always use a learning rate 5–10× smaller for the backbone than for the new head during fine-tuning.

**2. Not normalising inputs to match the pretraining distribution**  
Models pretrained on ImageNet expect inputs normalised with ImageNet statistics (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225] per channel). Using different normalisation produces out-of-distribution activations in the pretrained layers, degrading transferred features.

**3. Unfreezing all layers immediately with a small dataset**  
With very few target samples, unfreezing the full backbone produces severe overfitting — the model has enough capacity to memorise the small dataset. Always start with the backbone frozen, verify the head trains well, then unfreeze gradually from the top layers down.

**4. Using `model.eval()` mode during fine-tuning**  
Calling `model.eval()` freezes BatchNorm statistics and disables dropout for the entire model, including the layers you intend to fine-tune. Always call `model.train()` during the fine-tuning phase; switch to `model.eval()` only for evaluation.

**5. Assuming features transfer equally well across all domains**  
ImageNet features transfer well to natural images but may transfer poorly to medical images, satellite imagery, or scientific sensor data with very different statistical properties. Always benchmark fine-tuned transfer learning against training from scratch — transfer learning does not always win on domain-shifted data.
---
*python_methods_library - Samantha McGarrigle*